In [1]:
import torch
import pandas as pd

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DATA_DIR = Path('../data')
IMAGE_DIR = DATA_DIR / "train_images_224"

train_df = pd.read_csv(DATA_DIR / "train.csv")


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device", device)
print("데이터:", train_df.shape)

Device cuda
데이터: (18632, 2)


In [2]:
# Multi-hot Encoing

classes = [
    "scab",
    "healthy",
    "frog_eye_leaf_spot",
    "complex",
    "rust",
    "powdery_mildew"
]

def multi_hot_encode(labels):
    label_list = labels.split()

    encoded = []
    for cls in classes:
        if cls in label_list:
            encoded.append(1)
        else:
            encoded.append(0)
    return encoded

In [3]:
train_data = pd.read_csv(DATA_DIR/"train_split.csv")
valid_data = pd.read_csv(DATA_DIR/"valid_split.csv" )

print("Train:", len(train_data))
print("Valid:", len(valid_data))

Train: 14906
Valid: 3726


In [4]:

class PlanetDataset(Dataset):
    def __init__(self, df, image_dir, transform = None):
        self.df = df.reset_index(drop = True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx): # Dataset에서 이미지 한 장을 요청하면 해당 이미지와 정답을 반환하는 역할
        row = self.df.iloc[idx]

        image_path = self.image_dir / row["image"]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            multi_hot_encode(row["labels"]),
            dtype = torch.float32
        )

        return image, label

In [5]:
# Transform 정의

transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [6]:
# Dataset과 DataLoader 생성

train_dataset = PlanetDataset(
    train_data,
    IMAGE_DIR,
    transform = transform
)

valid_dataset = PlanetDataset(
    valid_data,
    IMAGE_DIR,
    transform = transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size = 32,
    shuffle = False,
    num_workers = 0
)

In [7]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Labels dtype:", labels.dtype)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 6])
Labels dtype: torch.float32


In [8]:
# ResNet18로 학습

# ResNet18의 구조와 사전학습된 가중치
# 마지막 분류기를 Linear(512, 6)으로 변경하는 이유
# Freeze와 Xavier Initialization을 언제 사용하는가


# ResNet18 불러오기
# ImageNet 데이터셋의 1,000개의 클래스를 분류하도록 사전학습된 모델이기 때문에 출력이 1000개

import torch.nn as nn
from torchvision import models

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights = weights)

print(model.fc)

Linear(in_features=512, out_features=1000, bias=True)


In [9]:
model.fc = nn.Linear(512, 6)
print(model.fc)

Linear(in_features=512, out_features=6, bias=True)


In [10]:
# Freeze란 무엇인가?
# 사전학습된 ResNet18은 이미 일반적인 이미지에서 유용한 시각적 특징을 학습한 상태
# 우리가 처음부터 모든 가중치를 변경할 필요가 없음
# 그래서 처음에는 기존 특징 추출기의 가중치를 고정, 새로 추가한 분류기만 학습


# 이렇게 하면 기존 ResNet18의 가중치는 Gradient를 계산하지 않고, 새로 생성한 fc의 가중치만 학습하게 됨
# requires_grad = False는 가중치 업데이트를 막는 것이지 BatchNorm의 Running Mean과 Variance까지 자동으로 고정하는 것은 아님
# 완전히 고정된 특징 추출기로 실험하려면 학습 시 BatchNorm 상태도 고려해야 됨


# Xavier Intialization은 지금 필요할까?
# model.Linear(512, 6)이 코드를 실행하면 PyTorch는 새로운 Linear Layer의 가중치를 기본 초기화 방식으로 자동 초기화
# 반드시 Xavier 초기화를 적용해야 하는 것은 아님


for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(512, 6)

In [11]:
# 모델 출력 확인하기

model = model.to(device)
images = images.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)

print("Output Shape:", outputs.shape)
print("Output Example:", outputs[0])

Output Shape: torch.Size([32, 6])
Output Example: tensor([-0.4926,  0.4855, -0.2547,  0.4593,  0.3188, -1.0053], device='cuda:0')


In [12]:
# 모델 학습

import torch
import torch.nn as nn
from torchvision import models

torch.manual_seed(42)

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights = weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(
    model.fc.in_features,   # model.fc.parameters()를 사용하는 이유는 현재 기존 ResNet18의 특징 추출기를 Freeze했기 때문에
                            # 새로운 분류기의 가중치만 업데이트하면 되기 때문
                            # model.parameters()를 사용해도 requires_grad = False인 파라미터는 일반적으로 업데이트 되지 않지만, 학습 대상을 명확하게 표현하기 위해서 작성                
    6
)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr = 0.001,
    weight_decay=0.01
)

In [13]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):  # ResNet18에서 Batch Normalization 계측이 학습 중에 사용하는 통계값을 업데이트하지 않도록 설정하는 코드
            module.eval()

    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # 1. Gradient 초기화
        optimizer.zero_grad()

        # 2. Forward
        outputs = model(images)
        
        # 3. Loss 계산
        loss = criterion(outputs, labels)

        # 4. Backward
        loss.backward()

        # 5. 가중치 업데이트
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

In [14]:
# Validation 함수 구현

from sklearn.metrics import f1_score
import numpy as np

def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).int()

            total_loss += loss.item() * images.size(0)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    valid_loss = total_loss / len(loader.dataset)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    valid_f1 = f1_score(
        all_labels,
        all_preds,
        average = "macro",
        zero_division=0
    )

    return valid_loss, valid_f1

In [15]:
EPOCHS = 5

train_losses = []
valid_losses = []
valid_f1_scores = []

best_f1 = -1

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_f1 = validate(
        model,
        valid_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)
    valid_f1_scores.append(valid_f1)

    print(
        f"Epoch [{epoch+1} / {EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Valid Loss: {valid_loss:.4f} "
        f"Valid F1: {valid_f1:.4f}"
    )

    if valid_f1 > best_f1:
        best_f1 = valid_f1

        torch.save(
            model.state_dict(),
            "best_resnet18.pth"
        )

print(f"\nBest Valid Macro F1: {best_f1:.4f}")

Epoch [1 / 5] Train Loss: 0.3412 Valid Loss: 0.2952 Valid F1: 0.5297
Epoch [2 / 5] Train Loss: 0.2813 Valid Loss: 0.2752 Valid F1: 0.5832
Epoch [3 / 5] Train Loss: 0.2654 Valid Loss: 0.2672 Valid F1: 0.6079
Epoch [4 / 5] Train Loss: 0.2575 Valid Loss: 0.2605 Valid F1: 0.6429
Epoch [5 / 5] Train Loss: 0.2523 Valid Loss: 0.2511 Valid F1: 0.6452

Best Valid Macro F1: 0.6452


In [16]:
# baseline 학습 결과 분석

# 1. Train Loss와 Valid Loss가 함께 감소하고 있음
# 2. Validation F1도 지속적으로 개선


# 이제 해야 할 것
# 1. 클래스별 성능 부터 분석
# Macro F1 0.6452는 6개의 클래스의 F1을 평균한 값이기 때문


In [17]:
# 클래스별 Precision / Recall / F1 확인

from sklearn.metrics import classification_report
import numpy as np

model.load_state_dict(
    torch.load(
        "best_resnet18.pth",
        map_location = device,
        weights_only = True
    )
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        
        outputs = model(images)

        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).int()

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

print(
    classification_report(
        all_labels,
        all_preds,
        target_names = classes,
        zero_division = 0
    )
)

                    precision    recall  f1-score   support

              scab       0.77      0.58      0.66      1123
           healthy       0.86      0.71      0.78       939
frog_eye_leaf_spot       0.71      0.46      0.56       893
           complex       0.68      0.28      0.40       432
              rust       0.89      0.61      0.72       407
    powdery_mildew       0.94      0.63      0.75       252

         micro avg       0.80      0.56      0.66      4046
         macro avg       0.81      0.55      0.65      4046
      weighted avg       0.79      0.56      0.65      4046
       samples avg       0.58      0.58      0.57      4046



In [18]:
# 클래스별 성능 분석

# Powdery mildew는 데이터가 가장 작은 클래스인데도 F1-score가 0.75
# 반면 Complex는 Powdery mildew보다 검증 데이터의 정답 수가 많지만, F1-score는 0.40에 불과

# Recall
# Macro percision은 0.81인데, Macro Recall은 0.55
# 모델이 질병이 있다고 예측할 때는 비교적 정확하지만, 실제로 존재하는 질병을 놓치는 경우가 많음
# 특히 Complex의 Recall은 0.28

# 여기서는 Unfreeze를 하는 방법도 있지만, Threshold를 변경하는 실험부터 진행하는 것이 좋아보임
# 현재 모델의 precision은 높은데 Recall이 낮기 때문

In [20]:
# Threshold 실험 진행

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels, in valid_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.sigmoid(outputs)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

In [22]:
# Threshold별 성능 비교

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

for threshold in thresholds:
    preds = (all_probs >= threshold).astype(int)

    precision = precision_score(
        all_labels, preds,
        average = "macro",
        zero_division = 0
    )

    recall = recall_score(
        all_labels, preds,
        average = "macro",
        zero_division = 0
    )

    f1 = f1_score(
        all_labels, preds,
        average = "macro",
        zero_division=0
    )

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Threshold: 0.2 | Precision: 0.6093 | Recall: 0.8042 | F1: 0.6856
Threshold: 0.3 | Precision: 0.6941 | Recall: 0.7171 | F1: 0.6998
Threshold: 0.4 | Precision: 0.7498 | Recall: 0.6319 | F1: 0.6813
Threshold: 0.5 | Precision: 0.8070 | Recall: 0.5451 | F1: 0.6452
Threshold: 0.6 | Precision: 0.8460 | Recall: 0.4536 | F1: 0.5811
Threshold: 0.7 | Precision: 0.8940 | Recall: 0.3730 | F1: 0.5121


In [24]:
# Threshold 0.3이 최적인가?

# 진짜 최적값은 아니다.
# 첫째, 우리는 0.2와 0.3 사이의 값이나 0.3과 0.4 사이의 값을 시험하지 않았다.
# 지금은 모든 클래스에 동일한 Threshold를 적용하고 있다.

# 다음 실험은 클래스별 Threshold 변화 분석

from sklearn.metrics import f1_score
import pandas as pd

results = []

for threshold in [0.3, 0.5]:
    preds = (all_probs >= threshold).astype(int)

    scores = f1_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    results.append(scores)

df_result = pd.DataFrame({
    "Class": classes,
    "F1 (Threshold 0.5)": results[1],
    "F1 (Threshold 0.3)": results[0]
})


df_result["Difference"] = (
    df_result["F1 (Threshold 0.3)"]
    - df_result["F1 (Threshold 0.5)"]
)

print(df_result.round(4))

                Class  F1 (Threshold 0.5)  F1 (Threshold 0.3)  Difference
0                scab              0.6619              0.6943      0.0323
1             healthy              0.7760              0.7872      0.0112
2  frog_eye_leaf_spot              0.5602              0.6443      0.0841
3             complex              0.3987              0.4975      0.0988
4                rust              0.7212              0.7683      0.0471
5      powdery_mildew              0.7536              0.8070      0.0535


In [25]:
# 결론 : 기존 Threshold 0.5는 현재 모델에서 다소 보수적인 결정 기준
# Threshold를 낮추는 것만으로 해결되지 않은 문제가 남아 있음

# 다음 단계는 Unfreeze를 통한 Fine-tuning

# 같은 데이터 분할 사용
# 같은 224 * 224 이미지 캐시 사용
# 기존 Baseline의 최고 성능 가중치에서 시작
# layer4와 fc만 학습
# Learning Rate는 기존보다 작게 설정
# Threshold 0.5를 기준으로 먼저비교하고, 이후 0.3에서도 비교



# layer4를 Unfreeze한다고 해서 반드시 성능이 좋아지는 것은 아님
# 사전학습된 특징을 변경하면서 오히려 기존에 유용했던 정보가 손상될 수도 있기 때문
# 이를 방지하기 위해 보통 Fine-tuning에서는 더 작은 Learning Rate를 사용하기도 해